In [9]:
import os
from typing import Annotated
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()
assert os.getenv("GROQ_API_KEY")
from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [10]:
llm = ChatGroq(model="qwen/qwen3-32b", reasoning_format="parsed")

In [14]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot(state: State) -> State:
    return {"messages": [llm.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()

In [15]:
message = {"role": "user", "content": "Who walked on the moon for the first time? Print only the name"}
# message = {"role": "user", "content": "What is the latest price of MSFT stock?"}
response = graph.invoke({"messages":[message]})

response["messages"]

[HumanMessage(content='Who walked on the moon for the first time? Print only the name', additional_kwargs={}, response_metadata={}, id='f21bce8b-20a0-43b8-90e1-a3c40945a882'),
 AIMessage(content='Neil Armstrong', additional_kwargs={'reasoning_content': 'Okay, I need to figure out who was the first person to walk on the moon. Let me start by recalling what I know about the space race. The United States and the Soviet Union were the main competitors during the Cold War, trying to achieve milestones in space exploration. The Apollo program was the U.S. initiative to land humans on the moon.\n\nI remember that Apollo 11 was the mission that successfully landed on the moon. The astronauts involved were Neil Armstrong, Buzz Aldrin, and Michael Collins. Michael Collins stayed in the command module orbiting the moon, while Armstrong and Aldrin descended to the lunar surface.\n\nSo, the first person to step onto the moon would be either Armstrong or Aldrin. I think there\'s a famous quote about

In [ ]:
state = None
while True:
    in_message = input("You: ")
    if in_message.lower() in {"quit","exit"}:
        break
    if state is None:
        state: State = {
            "messages": [{"role": "user", "content": in_message}]
        }
    else:
        state["messages"].append({"role": "user", "content": in_message})

    state = graph.invoke(state)
    print("Bot:", state["messages"][-1].content)